In [30]:
from twikit import Client, TooManyRequests, Forbidden
import time
from datetime import datetime
import configparser
from random import randint
import asyncio

In [31]:
minimum_tweet = 10
QUERY = 'Apple Vision Pro OR #VisionPro -is:retweet lang:en'
COOKIES_FILE = 'cookies.json'

In [32]:
# --- 1. CONFIG AND CREDENTIALS (Synchronous) ---
config = configparser.ConfigParser()
config.read('config.ini')

try:
    username = config['X']['username']
    password = config['X']['password']
    email = config['X']['email']
except KeyError:
    print("FATAL: Ensure your config.ini has a section named [X] with username, password, and email.")
    username = password = email = None

In [35]:
# ------------------------------------------------------------------
# --- 2. ASYNCHRONOUS SESSION SETUP AND FETCH FUNCTION ---
# ------------------------------------------------------------------

async def setup_client_and_fetch():
    """Initializes the client, handles login/cookies, and fetches the tweets."""
    # Authenticate to X.com (Client initialization is synchronous)
    client = Client(language='en-US')
    
    # 1. ATTEMPT TO LOAD COOKIES
    try:
        client.load_cookies(COOKIES_FILE)
        print("Cookies loaded successfully.")
    except FileNotFoundError:
        print("Cookies file not found. Logging in...")
        
        # 2. PERFORM ASYNCHRONOUS LOGIN IF COOKIES ARE MISSING (or expired/invalid)
        if not username or not password:
             print("Cannot log in: Credentials missing from config.ini.")
             return None
             
        try:
            # client.login MUST be awaited because it performs network I/O
            await client.login(auth_info_1=username, auth_info_2=email, password=password)
            client.save_cookies(COOKIES_FILE)
            print("Login successful and new cookies saved.")
        except Exception as e:
            print(f"FATAL AUTH ERROR: Could not log in. Check credentials and 2FA status. Details: {e}")
            return None

    # 3. PERFORM ASYNCHRONOUS DATA FETCH
    print(f"Fetching tweets for query: {QUERY}")
    try:
        # The client.search_tweet function MUST be awaited
        # Using 'Latest' is often more reliable than 'Top' for scraping
        tweets = await client.search_tweet(QUERY, product='Latest') 
        
        # 4. PROCESS RESULTS
        print(f"\nSuccessfully retrieved {len(tweets)} tweets on the first page.")
        if tweets:
            print("First Tweet Text (Sample):")
            # The 'text' attribute is usually available directly on the object
            print(getattr(tweets[0], 'text', 'Text attribute not found.')) 
            
        return tweets
    
    except Forbidden as e:
        print(f"\nCRITICAL ERROR (403 Forbidden): The current session is blocked or invalid for searching. Session has expired.")
        print(f"ACTION: Manually delete {COOKIES_FILE} and run again to force a fresh login.")
        return None
    except TooManyRequests:
        print("\nAPI Limit Reached. Wait for the reset.")
        return None
    except Exception as e:
        print(f"\nAn unexpected error occurred during fetch: {e}")
        return None

In [34]:
# ------------------------------------------------------------------
# --- 3. EXECUTION BLOCK ---
# ------------------------------------------------------------------
if __name__ == '__main__':
    # Execute the asynchronous function
    final_tweets = await setup_client_and_fetch()

Cookies file not found. Logging in...
FATAL AUTH ERROR: Could not log in. Check credentials and 2FA status. Details: status: 403, message: "<!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>
<title>Attention Required! | Cloudflare</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/cf.errors.css" />
<!--[if lt IE 9]><link rel="stylesheet" id='cf_styles-ie-css' href="/cdn-cgi/styles/cf.errors.ie.css" /><![endif]-->
<style>body{margin: